# Week 75, Day 5: Transactions and indexes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w075d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sqlite3, os

if os.path.exists("bank.db"):
    os.remove("bank.db")
setup = sqlite3.connect("bank.db")
with setup:
    setup.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER)")
    setup.execute("INSERT INTO accounts VALUES ('shop', 100)")

till_a, till_b = sqlite3.connect("bank.db"), sqlite3.connect("bank.db")

a = till_a.execute("SELECT balance FROM accounts WHERE name = 'shop'").fetchone()[0]
b = till_b.execute("SELECT balance FROM accounts WHERE name = 'shop'").fetchone()[0]
with till_a:
    till_a.execute("UPDATE accounts SET balance = ? WHERE name = 'shop'", (a + 30,))
with till_b:
    till_b.execute("UPDATE accounts SET balance = ? WHERE name = 'shop'", (b + 50,))

print(setup.execute("SELECT balance FROM accounts").fetchone()[0])

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("CREATE TABLE docs (id INTEGER PRIMARY KEY, title TEXT, version INTEGER NOT NULL DEFAULT 1); INSERT INTO docs (title) VALUES ('Draft');")

def rename(conn, doc_id, new_title, seen_version):
    with conn:
        cur = conn.execute("UPDATE docs SET title = ?, version = version + 1 WHERE id = ? AND version = ?",
                           (new_title, doc_id, seen_version))
    return cur.rowcount == 1

title, version = conn.execute("SELECT title, version FROM docs WHERE id = 1").fetchone()
print(rename(conn, 1, "Plan", version))         # first editor wins
print(rename(conn, 1, "Notes", version))        # second editor's version is stale: refused
print(conn.execute("SELECT title, version FROM docs").fetchone())

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT, joined TEXT)")
conn.executemany("INSERT INTO customers (name, city, joined) VALUES (?, ?, ?)",
                 [(f"c{i}", ["Leeds", "London", "Bristol"][i % 3], f"2025-{i % 12 + 1:02d}-01") for i in range(3000)])

def plan(sql):
    return [row[3] for row in conn.execute("EXPLAIN QUERY PLAN " + sql)]

query = "SELECT name FROM customers WHERE city = 'Leeds'"
print(plan(query))
conn.execute("CREATE INDEX customers_city ON customers (city)")
print(plan(query))
print(plan("SELECT name FROM customers WHERE lower(city) = 'leeds'"))
print(plan("SELECT name FROM customers WHERE id = 42"))

> **Key idea.** ACID: atomic, consistent, isolated, durable. Isolation prevents dirty reads, non-repeatable reads, phantoms and lost updates to varying degrees; SQLite allows one writer at a time (WAL lets readers continue), and a waiting writer times out with "database is locked". Prevent lost updates by doing the arithmetic in one UPDATE (with conditions, checking rowcount), by locking first (BEGIN IMMEDIATE, SELECT ... FOR UPDATE), or optimistically with a version column. Savepoints roll back part of a transaction. Indexes are sorted B-trees that turn scans into searches; check with EXPLAIN QUERY PLAN; expressions on a column and leading-wildcard LIKE can't use a plain index. Composite indexes follow the leftmost-prefix rule (equality columns first, then a range or sort column); covering indexes answer queries alone. Indexes slow writes and take space, so index for real, frequent queries, including foreign keys.

### Exercise: Reading query plans

Write `plan(conn, sql, params=())`: the list of detail strings from `EXPLAIN QUERY PLAN` (the last column of each row). Then `index_used(conn, sql, params=())`: the name of the first index the plan uses (from text such as `USING INDEX name` or `USING COVERING INDEX name`), `"primary key"` if it searches by the integer primary key, or `None` if it only scans. (The wording differs a little between SQLite versions, so match the key words with a regular expression rather than whole strings.)

In [ ]:
import re, sqlite3

def plan(conn, sql, params=()):
    ___

def index_used(conn, sql, params=()):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_plans():\n    \"SCAN or SEARCH\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"CREATE TABLE p (id INTEGER PRIMARY KEY, category TEXT, price REAL, name TEXT); CREATE INDEX p_category ON p (category);\")\n    assert index_used(conn, \"SELECT * FROM p WHERE name = ?\", (\"Tea\",)) is None\n    assert index_used(conn, \"SELECT * FROM p WHERE category = ?\", (\"food\",)) == \"p_category\"\n    assert index_used(conn, \"SELECT * FROM p WHERE id = 3\") == \"primary key\"\n    details = plan(conn, \"SELECT * FROM p WHERE name = 'x'\")\n    assert len(details) == 1 and details[0].startswith(\"SCAN\")")

### Exercise: Indexes for a workload

The shop's three slowest queries are in `QUERIES`. Write `INDEXES`: a list of `CREATE INDEX` statements (at most **three**) such that every query uses an index (no plain scans), and the third is answered by a **covering** index. Name each index after its table and columns, such as `orders_customer_id`.

In [ ]:
QUERIES = [
    "SELECT id, placed FROM orders WHERE customer_id = ?",
    "SELECT * FROM order_items WHERE product_id = ?",
    "SELECT price FROM products WHERE category = ? ORDER BY price",
]
INDEXES = [
    ___
]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_workload():\n    \"every query searches an index\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"\"\"\n    CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT);\n    CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL);\n    CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, placed TEXT);\n    CREATE TABLE order_items (order_id INTEGER, product_id INTEGER, qty INTEGER, PRIMARY KEY (order_id, product_id));\n    \"\"\")\n    assert len(INDEXES) <= 3\n    for statement in INDEXES:\n        conn.execute(statement)\n    for sql in QUERIES:\n        details = [r[-1] for r in conn.execute(\"EXPLAIN QUERY PLAN \" + sql, (1,))]\n        assert any(\"INDEX\" in d for d in details), (sql, details)\n        assert not any(\"TEMP B-TREE\" in d for d in details), (\"sorting without the index\", sql, details)\n    third = [r[-1] for r in conn.execute(\"EXPLAIN QUERY PLAN \" + QUERIES[2], (\"food\",))]\n    assert any(\"COVERING INDEX\" in d for d in third), third")

### Exercise: The leftmost-prefix rule

Write `usable_columns(index, equal, ranged=None)`: given an index's columns (a list, in order), the set of columns a query compares with `=`, and optionally one column it compares with a range (`<`, `>`, `BETWEEN`), return how many of the index's leading columns the query can use: walk the index's columns in order, counting each one in `equal`; then, at the first column not in `equal`, count it too if it's the `ranged` column, and stop. Then `best_index(indexes, equal, ranged=None)`: the name of the index (from a dict of name to columns) that can use the most columns (ties to the name first alphabetically), or `None` if none can use any.

In [ ]:
def usable_columns(index, equal, ranged=None):
    ___

def best_index(indexes, equal, ranged=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_prefix():\n    \"phone-book order\"\n    assert usable_columns([\"category\", \"price\"], {\"category\"}, \"price\") == 2\n    assert usable_columns([\"price\", \"category\"], {\"category\"}, \"price\") == 1\n    assert usable_columns([\"category\", \"price\"], {\"price\"}) == 0\n    assert usable_columns([\"a\", \"b\", \"c\"], {\"a\", \"c\"}) == 1 and usable_columns([\"a\", \"b\", \"c\"], {\"a\", \"b\", \"c\"}) == 3\n    assert usable_columns([\"a\", \"b\", \"c\"], {\"a\"}, \"b\") == 2\n    indexes = {\"by_cat\": [\"category\"], \"by_cat_price\": [\"category\", \"price\"], \"by_price\": [\"price\"]}\n    assert best_index(indexes, {\"category\"}, \"price\") == \"by_cat_price\"\n    assert best_index(indexes, {\"category\"}) == \"by_cat\" and best_index(indexes, set(), \"price\") == \"by_price\"\n    assert best_index(indexes, {\"name\"}) is None")

### Exercise: Withdrawals that can't be lost

Write `withdraw(conn, name, amount)`: in **one** `UPDATE` statement, take `amount` from the account only if it has enough (the condition is in the same statement's `WHERE`), commit, and return `True` if it happened, `False` if there wasn't enough money; raise `KeyError` if there's no such account (check only when the update changed nothing). Two "tills" each holding their own connection must never lose an update or overdraw the account.

In [ ]:
import sqlite3

def withdraw(conn, name, amount):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_atomic():\n    \"the database does the arithmetic\"\n    import sqlite3, os\n    if os.path.exists(\"till.db\"):\n        os.remove(\"till.db\")\n    setup = sqlite3.connect(\"till.db\")\n    with setup:\n        setup.execute(\"CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER NOT NULL)\")\n        setup.execute(\"INSERT INTO accounts VALUES ('shop', 100)\")\n    a, b = sqlite3.connect(\"till.db\"), sqlite3.connect(\"till.db\")\n    assert withdraw(a, \"shop\", 30) is True and withdraw(b, \"shop\", 50) is True\n    assert withdraw(a, \"shop\", 30) is False and withdraw(b, \"shop\", 20) is True\n    assert setup.execute(\"SELECT balance FROM accounts\").fetchone() == (0,)\n    try:\n        withdraw(a, \"nobody\", 1)\n        assert False\n    except KeyError:\n        pass\n    body = source.split(\"def withdraw\")[1]\n    assert body.count(\"execute(\") <= 2 and \"SELECT balance\" not in body, \"do the check and the change in one UPDATE\"")

### Exercise: Optimistic locking

A wiki keeps pages in `pages (id INTEGER PRIMARY KEY, title TEXT, body TEXT, version INTEGER NOT NULL)`. Write `load(conn, page_id)`: `(title, body, version)`. Then `save(conn, page_id, title, body, seen_version)`: update the page only if its version is still `seen_version`, incrementing the version, and return the new version; if someone saved in between, raise `Conflict` (given) with the page's current version as its `current` attribute. Finally `save_with_merge(conn, page_id, edit)`: load, apply `edit(title, body)` (returning the new title and body), save, and on a conflict start again from a fresh load, up to 3 attempts in all (then let the `Conflict` propagate).

In [ ]:
import sqlite3

class Conflict(Exception):
    def __init__(self, current):
        super().__init__(f"page changed; it's now at version {current}")
        self.current = current

def load(conn, page_id):
    ___

def save(conn, page_id, title, body, seen_version):
    ___

def save_with_merge(conn, page_id, edit):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_optimistic():\n    \"versions instead of locks\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.executescript(\"CREATE TABLE pages (id INTEGER PRIMARY KEY, title TEXT, body TEXT, version INTEGER NOT NULL); INSERT INTO pages VALUES (1, 'Tea', 'Hot.', 1);\")\n    title, body, v = load(conn, 1)\n    assert save(conn, 1, title, body + \" Lovely.\", v) == 2\n    try:\n        save(conn, 1, \"Coffee\", body, v)\n        assert False\n    except Conflict as e:\n        assert e.current == 2\n    assert load(conn, 1) == (\"Tea\", \"Hot. Lovely.\", 2)\n    calls = []\n    def sneaky_edit(title, body):\n        calls.append(body)\n        if len(calls) == 1:\n            with conn:\n                conn.execute(\"UPDATE pages SET body = body || ' Strong.', version = version + 1 WHERE id = 1\")\n        return title, body + \" With milk.\"\n    assert save_with_merge(conn, 1, sneaky_edit) == 4\n    assert load(conn, 1)[1] == \"Hot. Lovely. Strong. With milk.\" and len(calls) == 2\n    def always_raced(title, body):\n        with conn:\n            conn.execute(\"UPDATE pages SET version = version + 1 WHERE id = 1\")\n        return title, body\n    try:\n        save_with_merge(conn, 1, always_raced)\n        assert False\n    except Conflict:\n        pass")

### Exercise: Keeping the good rows

Write `import_people(conn, people)`: insert every person (a dict with `"email"` and `"age"`) into `people (email TEXT PRIMARY KEY, age INTEGER NOT NULL CHECK (age >= 0))` inside **one** transaction, but give each row its own **savepoint**, so a row that breaks a constraint is rolled back on its own while the others are kept. Commit at the end, and return `(added, rejected)`: the number added and the list of rejected emails, in order. Begin the transaction explicitly with `BEGIN`.

In [ ]:
import sqlite3

def import_people(conn, people):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_savepoints():\n    \"partial rollback\"\n    import sqlite3\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"CREATE TABLE people (email TEXT PRIMARY KEY, age INTEGER NOT NULL CHECK (age >= 0))\")\n    people = [{\"email\": \"ada@x\", \"age\": 36}, {\"email\": \"bob@x\", \"age\": -1}, {\"email\": \"ada@x\", \"age\": 40},\n              {\"email\": \"cy@x\", \"age\": None}, {\"email\": \"di@x\", \"age\": 28}]\n    assert import_people(conn, people) == (2, [\"bob@x\", \"ada@x\", \"cy@x\"])\n    assert conn.execute(\"SELECT * FROM people ORDER BY email\").fetchall() == [(\"ada@x\", 36), (\"di@x\", 28)]\n    assert not conn.in_transaction\n    assert \"SAVEPOINT\" in source.upper() and \"ROLLBACK TO\" in source.upper()")

### Bug hunt: Two tills, one lost payment

Each till reads the balance, adds its payment in Python, and writes the result back, so the second write overwrites the first and this prints `150`. Make each payment a single `UPDATE` that adds to the stored balance, so it prints `180`.

In [ ]:
import sqlite3, os

if os.path.exists("tills.db"):
    os.remove("tills.db")
setup = sqlite3.connect("tills.db")
with setup:
    setup.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER)")
    setup.execute("INSERT INTO accounts VALUES ('shop', 100)")
till_a, till_b = sqlite3.connect("tills.db"), sqlite3.connect("tills.db")

a = till_a.execute("SELECT balance FROM accounts WHERE name = 'shop'").fetchone()[0]
b = till_b.execute("SELECT balance FROM accounts WHERE name = 'shop'").fetchone()[0]
with till_a:
    till_a.execute("UPDATE accounts SET balance = ? WHERE name = 'shop'", (a + 30,))
with till_b:
    till_b.execute("UPDATE accounts SET balance = ? WHERE name = 'shop'", (b + 50,))
print(setup.execute("SELECT balance FROM accounts").fetchone()[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no lost update\"\n    expect_output(\"180\")")

### Bug hunt: A composite index in the wrong order

The index is on `(price, category)`, but the query filters on `category` alone, so by the leftmost-prefix rule it can't use the index, and this prints `scan`. Create the index with the columns the other way round, so it prints `index`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, category TEXT, price REAL)")
conn.execute("CREATE INDEX products_price_category ON products (price, category)")
details = [r[-1] for r in conn.execute("EXPLAIN QUERY PLAN SELECT name FROM products WHERE category = ? ORDER BY price", ("food",))]
print("index" if any("INDEX" in d and d.startswith("SEARCH") for d in details) else "scan")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"equality first\"\n    expect_output(\"index\")")

### Bug hunt: An index hidden by a function

Logins look users up with `lower(email) = ?`, and the index is on `email` itself, so the planner can't use it and this prints `scan`. Index the **expression** `lower(email)` instead, so it prints `index`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, email TEXT)")
conn.execute("CREATE INDEX users_email ON users (email)")
details = [r[-1] for r in conn.execute("EXPLAIN QUERY PLAN SELECT id FROM users WHERE lower(email) = ?", ("ada@x",))]
print("index" if any("INDEX" in d for d in details) else "scan")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an expression index\"\n    expect_output(\"index\")")

### Puzzle: A savepoint around a risky step

Arrange the lines that try one insert inside a larger transaction, rolling back just that insert if it breaks a constraint. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the ACID property that means "all of its changes happen, or none do"
q = ___   # True or False: an index on (category, price) helps WHERE price < 5 on its own
r = ___   # what the warm-up's balance would be with UPDATE ... SET balance = balance + ? for both payments
s = ___   # the plan word for reading every row of a table: "SCAN" or "SEARCH"
t = ___   # True or False: every extra index makes INSERTs slower

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"atomicity\" and q is False and r == 180, \"p, q and r\"\n    assert s == \"SCAN\" and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an index adviser

Write `advise(workload, existing)`. `workload` is a list of queries described as dicts: `"table"`, `"equal"` (a list of columns compared with `=`), optionally `"range"` (one column compared with a range) and `"order"` (a list of columns sorted by), and `"per_day"` (how often it runs). `existing` maps index names to `(table, columns)`. For each query, the **ideal** index is its equality columns (sorted alphabetically, since their order doesn't matter), followed by its range column if it has one, **otherwise** its order columns. A query is **already served** if some existing index (or one already recommended) on that table **starts with** its ideal columns. Recommend one index per distinct (table, ideal columns) not already served, named `TABLE_COL1_COL2...`, and return a list of `(name, table, columns, per_day)`, where `per_day` is the total of every query that index would serve, most-used first (ties by name). Recommend nothing for a query with no equality, range or order columns. Process the workload by total use (most-used ideal index first), so a wider recommendation can serve a narrower query that comes later.

In [ ]:
def advise(workload, existing):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_adviser():\n    \"turning a workload into indexes\"\n    workload = [\n        {\"table\": \"orders\", \"equal\": [\"customer_id\"], \"per_day\": 5000},\n        {\"table\": \"orders\", \"equal\": [\"status\", \"customer_id\"], \"order\": [\"placed\"], \"per_day\": 800},\n        {\"table\": \"products\", \"equal\": [\"category\"], \"range\": \"price\", \"per_day\": 3000},\n        {\"table\": \"products\", \"equal\": [\"category\"], \"per_day\": 1000},\n        {\"table\": \"customers\", \"equal\": [\"email\"], \"per_day\": 9000},\n        {\"table\": \"customers\", \"equal\": [], \"per_day\": 10},\n        {\"table\": \"order_items\", \"equal\": [\"order_id\"], \"per_day\": 4000},\n    ]\n    existing = {\"customers_email\": (\"customers\", [\"email\"]), \"order_items_pk\": (\"order_items\", [\"order_id\", \"product_id\"])}\n    assert advise(workload, existing) == [\n        (\"orders_customer_id\", \"orders\", [\"customer_id\"], 5000),\n        (\"products_category_price\", \"products\", [\"category\", \"price\"], 4000),\n        (\"orders_customer_id_status_placed\", \"orders\", [\"customer_id\", \"status\", \"placed\"], 800),\n    ]\n    assert advise([], {}) == []")

## Warm-up

Two tills each took a payment: 30 and 50. The balance should be 180, but it's 150. Both read 100, both computed a new balance in Python, and the second write overwrote the first. This is the **lost update**, and no error was raised: each statement was perfectly valid. Preventing it is the job of today's first half.

## ACID and isolation

A transaction (yesterday's `with conn:`) promises four things, known as **ACID**:

- **Atomicity**: all of its changes happen, or none do.
- **Consistency**: it takes the database from one valid state to another; constraints (`NOT NULL`, `CHECK`, foreign keys) hold at the end.
- **Isolation**: transactions running at the same time don't see each other's unfinished work, and the result is as if they'd run in **some** order.
- **Durability**: once committed, changes survive a crash or power cut (they're on disk).

Isolation is the subtle one. Without it, concurrent transactions suffer **anomalies**: a **dirty read** (seeing another transaction's uncommitted change, which may be rolled back), a **non-repeatable read** (reading a row twice in one transaction and getting different values), a **phantom** (re-running a query and finding new rows), and the **lost update**. Databases offer **isolation levels** that trade protection for concurrency: `READ COMMITTED` (PostgreSQL's default) prevents dirty reads; `REPEATABLE READ` and `SERIALIZABLE` prevent more, at the cost of transactions sometimes failing and needing a retry.

**SQLite** is simpler: one writer at a time. A transaction that writes takes a lock on the whole database file; other connections can keep reading (in **WAL** mode, `PRAGMA journal_mode = WAL`, even while it writes), but a second writer must wait, and gets `database is locked` if it waits longer than its **timeout** (`sqlite3.connect(path, timeout=5)`). So SQLite transactions are serializable. That's why the warm-up still lost an update: its reads happened **outside** any transaction, and the computation happened in Python, between them.

## Lost updates, and how to prevent them

Three ways to stop the warm-up's lost update, from simplest to most general:

1. **Let the database do the arithmetic**, in one statement: `UPDATE accounts SET balance = balance + ? WHERE name = ?`. The read and the write happen together, inside the database, so nothing can come between them. Add conditions to the same statement to enforce rules: `... SET balance = balance - ? WHERE name = ? AND balance >= ?`, then check `rowcount` to see whether it happened.
2. **Lock while you work.** Start the transaction **before** reading, as a writer: in SQLite, `BEGIN IMMEDIATE` takes the write lock at once, so no one else can write until you commit; in other databases, `SELECT ... FOR UPDATE` locks the rows you read.
3. **Optimistic locking**: give each row a `version` number. Read the row and its version; when writing, `UPDATE ... SET ..., version = version + 1 WHERE id = ? AND version = ?`. If someone else changed it meanwhile, the version won't match, `rowcount` is 0, and you know to reload and try again (or tell the user). No locks are held while the user thinks, which suits web applications.

**Savepoints** give a transaction checkpoints you can roll back to without abandoning the whole thing: `SAVEPOINT name`, then either `RELEASE name` (keep the work since then) or `ROLLBACK TO name` (undo it, and carry on). Importing a thousand rows where a few are bad: wrap each row in a savepoint, roll back just the bad ones, and commit the rest together.

## Indexes

Without help, finding the customers in Leeds means a **full table scan**: reading every row and checking it. That's fine for a thousand rows, and hopeless for a hundred million. An **index** is a separate, sorted structure (a **B-tree**, a balanced tree of sorted keys with pointers to the rows) that lets the database jump to matching rows in about log(n) steps, like a book's index:

```sql
CREATE INDEX customers_city ON customers (city);
```

- An index on `city` helps `WHERE city = 'Leeds'`, range conditions such as `WHERE joined > '2025-01-01'` (on an index of `joined`), `ORDER BY city` (the rows are already sorted), and joins on that column.
- Primary keys and `UNIQUE` columns are indexed automatically. **Foreign key columns are not**, in SQLite and PostgreSQL, though they're joined on constantly: index them.
- **`EXPLAIN QUERY PLAN`** before a query shows how SQLite will run it: `SCAN customers` (reading the whole table) or `SEARCH customers USING INDEX customers_city (city=?)`. Other databases have `EXPLAIN` (and `EXPLAIN ANALYZE`, which runs the query and reports real timings).

An index can't be used when the query hides the column inside an expression: `WHERE lower(email) = ?` can't use an index on `email`, because the index is sorted by `email`, not `lower(email)`. Store the value normalised, or create an **expression index**: `CREATE INDEX users_email ON users (lower(email))`. Similarly, `LIKE '%term'` (a leading wildcard) can't use an ordinary index.

## Choosing indexes

**Composite indexes** cover several columns, sorted by the first, then the second within it, and so on, like a phone book sorted by surname then first name. That gives the **leftmost-prefix rule**: an index on `(category, price)` helps queries on `category`, or on `category` **and** `price`, or `category = ? ORDER BY price`; but **not** a query on `price` alone, just as a phone book doesn't help you find everyone called "Ada". Put **equality** columns first and a **range** or sort column last: `WHERE category = ? AND price < ?` uses both columns of `(category, price)`, but only the first of `(price, category)`.

A **covering index** contains every column a query needs, so the database never visits the table itself: `SELECT price FROM products WHERE category = ?` is answered from `(category, price)` alone (`USING COVERING INDEX` in the plan).

Indexes aren't free:

- every `INSERT`, `UPDATE` and `DELETE` must update every index on the table, so writes slow down;
- they take disk space and memory;
- an index on a column with few distinct values (a yes/no flag) barely helps, because each value matches a large part of the table.

So index for the queries you actually run, especially the frequent and slow ones; check with `EXPLAIN QUERY PLAN` that they're used; and drop the ones that aren't. `ANALYZE` gathers statistics that help SQLite's planner choose between indexes.

## Practice

## On your own computer

Feel the difference an index makes on a big table, with Python's `timeit`:

```python
import sqlite3, random, timeit

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, email TEXT)")
conn.executemany("INSERT INTO t (email) VALUES (?)", ((f"user{i}@example.com",) for i in range(1_000_000)))
look = lambda: conn.execute("SELECT id FROM t WHERE email = ?", (f"user{random.randrange(1_000_000)}@example.com",)).fetchone()
print("scan:", timeit.timeit(look, number=20) / 20)
conn.execute("CREATE INDEX t_email ON t (email)")
print("index:", timeit.timeit(look, number=20) / 20)
```

Expect the indexed lookup to be thousands of times faster. Then time the `executemany` with the index already in place: inserts get slower.

## Quiz

## Daily challenge

## Before you go

Run `EXPLAIN QUERY PLAN` on the queries you wrote on Days 1 and 2, against the shop database, and decide which indexes they'd need on a big table. Tomorrow's Forge puts the week together: a library loans system with its schema, queries, transactions, indexes and migrations.